# USL Championship — Team Data

League-wide metrics using the ASA public API.

In [1]:
from typing import Final

import pandas as pd
from itscalledsoccer.client import AmericanSoccerAnalysis

pd.options.display.max_columns = 999
pd.set_option("expand_frame_repr", False)

asa_client = AmericanSoccerAnalysis()

LEAGUE: Final[str] = "uslc"
FOCUS_TEAM: Final[str] = "LOU"  # change to any team abbreviation to repoint the notebook

## Data Fetch

In [2]:
team_xg = asa_client.get_team_xgoals(
    leagues=[LEAGUE],
    split_by_seasons=True,
    stage_name="Regular Season",
)

team_xp = asa_client.get_team_xpass(
    leagues=[LEAGUE],
    split_by_seasons=True,
    stage_name="Regular Season",
)

team_ga = asa_client.get_team_goals_added(
    leagues=[LEAGUE],
    split_by_seasons=True,
    stage_name="Regular Season",
)

## Data Cleaning

Resolve `team_id` to abbreviation, then join xGoals, xPass, and Goals Added
into a single frame.


In [3]:
teams = asa_client.get_teams(leagues=[LEAGUE])
team_map: dict[str, str] = dict(zip(teams["team_id"], teams["team_abbreviation"]))

for _df in [team_xg, team_xp, team_ga]:
    _df["team_name"] = _df["team_id"].map(team_map)
    _df.drop(columns=["team_id"], inplace=True)

# team_ga arrives with a nested `data` column — a list of per-action-type dicts
# (Dribbling, Fouling, Interrupting, Passing, Receiving, Shooting, Claiming).
# Explode + json_normalize unpacks those into one row per (team, season, action),
# then pivot reshapes into ga_for_<action> / ga_against_<action> columns.
_ga_long = team_ga.explode("data", ignore_index=True)
_ga_long = _ga_long.join(pd.json_normalize(_ga_long.pop("data")))
team_ga_wide = _ga_long.pivot(
    index=["team_name", "season_name", "minutes"],
    columns="action_type",
    values=["goals_added_for", "goals_added_against"],
)
team_ga_wide.columns = [
    f"ga_{direction.rsplit('_', 1)[-1]}_{action.lower()}"
    for direction, action in team_ga_wide.columns
]
team_ga_wide = team_ga_wide.reset_index()

# Totals across all action types, plus net differential.
_ga_for_cols = [c for c in team_ga_wide.columns if c.startswith("ga_for_")]
_ga_against_cols = [c for c in team_ga_wide.columns if c.startswith("ga_against_")]
team_ga_wide["ga_for_total"] = team_ga_wide[_ga_for_cols].sum(axis=1)
team_ga_wide["ga_against_total"] = team_ga_wide[_ga_against_cols].sum(axis=1)
team_ga_wide["ga_difference"] = (
    team_ga_wide["ga_for_total"] - team_ga_wide["ga_against_total"]
)
_ga_numeric_cols = [c for c in team_ga_wide.columns if c.startswith("ga_")]
team_ga_wide[_ga_numeric_cols] = team_ga_wide[_ga_numeric_cols].round(2)

# Merge xPass then the expanded Goals Added frame into xGoals, each time keeping
# only source-exclusive columns to prevent suffix conflicts on shared columns.
_join_on = ["team_name", "season_name"]
for _source in [team_xp, team_ga_wide]:
    _cols = _join_on + [c for c in _source.columns if c not in team_xg.columns]
    team_xg = team_xg.merge(_source[_cols], on=_join_on, how="left")

team_stats = (
    team_xg.copy()
    .sort_values(["team_name", "season_name"])
    .reset_index(drop=True)
)

# team_stats.info()
display(team_stats.head())

,season_name,count_games,shots_for,shots_against,goals_for,goals_against,goal_difference,xgoals_for,xgoals_against,xgoal_difference,goal_difference_minus_xgoal_difference,points,xpoints,team_name,attempted_passes_for,pass_completion_percentage_for,xpass_completion_percentage_for,passes_completed_over_expected_for,passes_completed_over_expected_p100_for,avg_vertical_distance_for,attempted_passes_against,pass_completion_percentage_against,xpass_completion_percentage_against,passes_completed_over_expected_against,passes_completed_over_expected_p100_against,avg_vertical_distance_against,passes_completed_over_expected_difference,avg_vertical_distance_difference,minutes,ga_for_claiming,ga_for_dribbling,ga_for_fouling,ga_for_interrupting,ga_for_passing,ga_for_receiving,ga_for_shooting,ga_against_claiming,ga_against_dribbling,ga_against_fouling,ga_against_interrupting,ga_against_passing,ga_against_receiving,ga_against_shooting,ga_for_total,ga_against_total,ga_difference
0,2018,34,397,550,33,69,-36,35.0732,56.4947,-21.4215,-14.5785,31,34.610,ATL,17877,0.7995,0.8023,-51.4825,-0.2880,6.4737,15349,0.7589,0.7497,141.4291,0.9214,7.8179,-192.9115,-1.3442,3362,0.28,0.47,2.17,21.01,0.62,15.86,8.87,0.26,3.37,2.38,20.59,4.07,25.61,13.52,49.27,69.81,-20.54
1,2019,34,436,484,44,73,-29,46.3788,66.8307,-20.4520,-8.5480,35,36.010,ATL,17070,0.8124,0.8077,80.0463,0.4689,6.0488,14487,0.7701,0.7593,157.1070,1.0845,7.8137,-77.0608,-1.7649,3342,0.08,0.93,2.79,18.05,4.98,20.98,10.11,-0.08,7.63,4.38,23.28,4.34,23.53,12.10,57.91,75.19,-17.27
2,2020,16,172,255,22,33,-11,19.4202,33.5177,-14.0975,3.0975,12,13.868,ATL,6737,0.7766,0.7824,-39.3618,-0.5843,7.5965,7279,0.7839,0.7756,60.3546,0.8292,7.2324,-99.7164,0.3641,1604,0.11,-0.94,0.99,9.07,3.90,9.28,3.74,0.03,2.87,2.09,8.75,5.83,12.93,6.48,26.16,38.98,-12.82
3,2021,32,420,441,43,57,-14,47.6095,58.9908,-11.3813,-2.6187,31,37.171,ATL,16953,0.8098,0.8099,-1.7108,-0.0101,5.8928,13090,0.7477,0.7511,-43.6221,-0.3332,8.3272,41.9113,-2.4343,3171,-0.29,2.16,2.38,20.86,4.19,17.43,9.56,-0.29,2.54,2.55,20.41,5.98,21.93,11.99,56.28,65.11,-8.83
4,2022,34,346,659,38,83,-45,40.1673,85.7422,-45.5749,0.5749,23,25.498,ATL,15765,0.7874,0.7948,-115.5356,-0.7329,7.1082,16530,0.7923,0.7714,344.1131,2.0817,6.9170,-459.6487,0.1913,3360,0.17,-2.39,2.05,27.70,3.38,21.48,7.80,-0.33,4.63,4.02,20.01,9.83,35.88,16.92,60.18,90.95,-30.77


## Per-Game Metrics

Normalize cumulative totals by `count_games` so teams with different schedules
are directly comparable.

In [4]:
# Shots — per game
team_stats["shots_per_game"] = (
    team_stats["shots_for"] / team_stats["count_games"]
).round(2)
team_stats["shots_against_per_game"] = (
    team_stats["shots_against"] / team_stats["count_games"]
).round(2)

# Goals — per game
team_stats["goals_per_game"] = (
    team_stats["goals_for"] / team_stats["count_games"]
).round(2)
team_stats["goals_against_per_game"] = (
    team_stats["goals_against"] / team_stats["count_games"]
).round(2)

# xG — per game
team_stats["xg_per_game"] = (
    team_stats["xgoals_for"] / team_stats["count_games"]
).round(2)
team_stats["xga_per_game"] = (
    team_stats["xgoals_against"] / team_stats["count_games"]
).round(2)

# Overperformance vs expected (positive = better than model predicts)
team_stats["finishing_overperformance"] = (
    team_stats["goals_for"] - team_stats["xgoals_for"]
).round(2)
team_stats["defensive_overperformance"] = (
    team_stats["xgoals_against"] - team_stats["goals_against"]
).round(2)

# Points — per game, expected, overperformance
team_stats["pts_per_game"] = (
    team_stats["points"] / team_stats["count_games"]
).round(2)
team_stats["xpts_per_game"] = (
    team_stats["xpoints"] / team_stats["count_games"]
).round(2)
team_stats["points_overperformance"] = (
    team_stats["points"] - team_stats["xpoints"]
).round(2)

# Passes — per game
team_stats["passes_per_game"] = (
    team_stats["attempted_passes_for"] / team_stats["count_games"]
).round(1)

## Advanced Metrics

Derived analytical layers built on top of the per-game cell: shot quality,
possession share, g+ per 96 minutes, season-over-season deltas, and
within-season percentile rank.

In [5]:
# Shot quality — xG generated per shot taken/allowed. Separates teams that
# create good looks from teams that just shoot often.
team_stats["xg_per_shot_for"] = (
    team_stats["xgoals_for"] / team_stats["shots_for"]
).round(3)
team_stats["xg_per_shot_against"] = (
    team_stats["xgoals_against"] / team_stats["shots_against"]
).round(3)

# Possession share proxy — pass-volume share vs opponent. ASA does not
# publish possession %, but pass share is a strong proxy and pairs naturally
# with the existing vertical-distance metrics.
team_stats["pass_share"] = (
    team_stats["attempted_passes_for"]
    / (team_stats["attempted_passes_for"] + team_stats["attempted_passes_against"])
).round(3)

In [6]:
# g+ per 96 minutes — ASA's standard rate convention. Normalizes across
# seasons with different game counts (15-game 2020 vs 34-game 2024).
_ga_cols = [c for c in team_stats.columns if c.startswith("ga_")]
_per96 = 96 / team_stats["minutes"]
for _c in _ga_cols:
    team_stats[f"{_c}_p96"] = (team_stats[_c] * _per96).round(3)

In [7]:
# Season-over-season change per team. First season for each team yields NaN.
# Relies on team_stats being sorted by (team_name, season_name) — enforced in
# the Data Cleaning cell.
_delta_cols = [
    "pts_per_game", "xpts_per_game",
    "goals_per_game", "xg_per_game",
    "goals_against_per_game", "xga_per_game",
    "ga_difference",
]
for _c in _delta_cols:
    team_stats[f"{_c}_yoy"] = (
        team_stats.groupby("team_name")[_c].diff().round(2)
    )

In [8]:
# League percentile rank within season (0–1, higher = better). For defensive
# metrics (lower raw value is better), rank descending so the stingiest defense
# still scores near 1.0.
_rank_high_is_good = [
    "pts_per_game", "xpts_per_game",
    "goals_per_game", "xg_per_game",
    "xg_per_shot_for",
    "pass_share",
    "ga_for_total", "ga_difference",
]
_rank_low_is_good = [
    "goals_against_per_game", "xga_per_game",
    "xg_per_shot_against",
    "ga_against_total",
]
for _c in _rank_high_is_good:
    team_stats[f"{_c}_pct"] = (
        team_stats.groupby("season_name")[_c].rank(pct=True).round(3)
    )
for _c in _rank_low_is_good:
    team_stats[f"{_c}_pct"] = (
        team_stats.groupby("season_name")[_c]
        .rank(pct=True, ascending=False)
        .round(3)
    )

---

In [9]:
_col_order = [
    # Identity
    "team_name", "season_name", "count_games", "minutes",
    # Points
    "points", "xpoints", "pts_per_game", "xpts_per_game", "points_overperformance",
    "pts_per_game_pct", "xpts_per_game_pct",
    "pts_per_game_yoy", "xpts_per_game_yoy",
    # Scoring
    "goals_for", "goals_per_game", "xgoals_for", "xg_per_game",
    "finishing_overperformance", "shots_for", "shots_per_game",
    "xg_per_shot_for", "xg_per_shot_for_pct",
    "goals_per_game_pct", "xg_per_game_pct",
    "goals_per_game_yoy", "xg_per_game_yoy",
    # Conceded
    "goals_against", "goals_against_per_game", "xgoals_against", "xga_per_game",
    "defensive_overperformance", "shots_against", "shots_against_per_game",
    "xg_per_shot_against", "xg_per_shot_against_pct",
    "goals_against_per_game_pct", "xga_per_game_pct",
    "goals_against_per_game_yoy", "xga_per_game_yoy",
    # Differential
    "goal_difference", "xgoal_difference", "goal_difference_minus_xgoal_difference",
    # Passing — offensive
    "attempted_passes_for", "passes_per_game", "pass_share", "pass_share_pct",
    "pass_completion_percentage_for", "xpass_completion_percentage_for",
    "passes_completed_over_expected_for", "passes_completed_over_expected_p100_for",
    "avg_vertical_distance_for",
    # Passing — against
    "attempted_passes_against",
    "pass_completion_percentage_against", "xpass_completion_percentage_against",
    "passes_completed_over_expected_against", "passes_completed_over_expected_p100_against",
    "avg_vertical_distance_against",
    # Passing — difference
    "passes_completed_over_expected_difference", "avg_vertical_distance_difference",
    # Goals Added — totals and net
    "ga_for_total", "ga_against_total", "ga_difference",
    "ga_for_total_pct", "ga_against_total_pct", "ga_difference_pct",
    "ga_difference_yoy",
    # Goals Added — totals per 96
    "ga_for_total_p96", "ga_against_total_p96", "ga_difference_p96",
    # Goals Added — offensive by action type
    "ga_for_passing", "ga_for_receiving", "ga_for_dribbling", "ga_for_shooting",
    "ga_for_interrupting", "ga_for_fouling", "ga_for_claiming",
    # Goals Added — offensive by action type per 96
    "ga_for_passing_p96", "ga_for_receiving_p96", "ga_for_dribbling_p96",
    "ga_for_shooting_p96", "ga_for_interrupting_p96", "ga_for_fouling_p96",
    "ga_for_claiming_p96",
    # Goals Added — defensive by action type
    "ga_against_passing", "ga_against_receiving", "ga_against_dribbling",
    "ga_against_shooting", "ga_against_interrupting", "ga_against_fouling",
    "ga_against_claiming",
    # Goals Added — defensive by action type per 96
    "ga_against_passing_p96", "ga_against_receiving_p96", "ga_against_dribbling_p96",
    "ga_against_shooting_p96", "ga_against_interrupting_p96",
    "ga_against_fouling_p96", "ga_against_claiming_p96",
]
team_stats = team_stats[_col_order]
display(team_stats.head())

,team_name,season_name,count_games,minutes,points,xpoints,pts_per_game,xpts_per_game,points_overperformance,pts_per_game_pct,xpts_per_game_pct,pts_per_game_yoy,xpts_per_game_yoy,goals_for,goals_per_game,xgoals_for,xg_per_game,finishing_overperformance,shots_for,shots_per_game,xg_per_shot_for,xg_per_shot_for_pct,goals_per_game_pct,xg_per_game_pct,goals_per_game_yoy,xg_per_game_yoy,goals_against,goals_against_per_game,xgoals_against,xga_per_game,defensive_overperformance,shots_against,shots_against_per_game,xg_per_shot_against,xg_per_shot_against_pct,goals_against_per_game_pct,xga_per_game_pct,goals_against_per_game_yoy,xga_per_game_yoy,goal_difference,xgoal_difference,goal_difference_minus_xgoal_difference,attempted_passes_for,passes_per_game,pass_share,pass_share_pct,pass_completion_percentage_for,xpass_completion_percentage_for,passes_completed_over_expected_for,passes_completed_over_expected_p100_for,avg_vertical_distance_for,attempted_passes_against,pass_completion_percentage_against,xpass_completion_percentage_against,passes_completed_over_expected_against,passes_completed_over_expected_p100_against,avg_vertical_distance_against,passes_completed_over_expected_difference,avg_vertical_distance_difference,ga_for_total,ga_against_total,ga_difference,ga_for_total_pct,ga_against_total_pct,ga_difference_pct,ga_difference_yoy,ga_for_total_p96,ga_against_total_p96,ga_difference_p96,ga_for_passing,ga_for_receiving,ga_for_dribbling,ga_for_shooting,ga_for_interrupting,ga_for_fouling,ga_for_claiming,ga_for_passing_p96,ga_for_receiving_p96,ga_for_dribbling_p96,ga_for_shooting_p96,ga_for_interrupting_p96,ga_for_fouling_p96,ga_for_claiming_p96,ga_against_passing,ga_against_receiving,ga_against_dribbling,ga_against_shooting,ga_against_interrupting,ga_against_fouling,ga_against_claiming,ga_against_passing_p96,ga_against_receiving_p96,ga_against_dribbling_p96,ga_against_shooting_p96,ga_against_interrupting_p96,ga_against_fouling_p96,ga_against_claiming_p96
0,ATL,2018,34,3362,31,34.610,0.91,1.02,-3.61,0.167,0.152,NaN,NaN,33,0.97,35.0732,1.03,-2.07,397,11.68,0.088,0.030,0.091,0.121,NaN,NaN,69,2.03,56.4947,1.66,-12.51,550,16.18,0.103,0.485,0.167,0.182,NaN,NaN,-36,-21.4215,-14.5785,17877,525.8,0.538,0.848,0.7995,0.8023,-51.4825,-0.2880,6.4737,15349,0.7589,0.7497,141.4291,0.9214,7.8179,-192.9115,-1.3442,49.27,69.81,-20.54,0.030,0.152,0.061,NaN,1.407,1.993,-0.587,0.62,15.86,0.47,8.87,21.01,2.17,0.28,0.018,0.453,0.013,0.253,0.600,0.062,0.008,4.07,25.61,3.37,13.52,20.59,2.38,0.26,0.116,0.731,0.096,0.386,0.588,0.068,0.007
1,ATL,2019,34,3342,35,36.010,1.03,1.06,-1.01,0.222,0.167,0.12,0.04,44,1.29,46.3788,1.36,-2.38,436,12.82,0.106,0.306,0.264,0.403,0.32,0.33,73,2.15,66.8307,1.97,-6.17,484,14.24,0.138,0.056,0.139,0.111,0.12,0.31,-29,-20.4520,-8.5480,17070,502.1,0.541,0.861,0.8124,0.8077,80.0463,0.4689,6.0488,14487,0.7701,0.7593,157.1070,1.0845,7.8137,-77.0608,-1.7649,57.91,75.19,-17.27,0.222,0.167,0.139,3.27,1.663,2.160,-0.496,4.98,20.98,0.93,10.11,18.05,2.79,0.08,0.143,0.603,0.027,0.290,0.518,0.080,0.002,4.34,23.53,7.63,12.10,23.28,4.38,-0.08,0.125,0.676,0.219,0.348,0.669,0.126,-0.002
2,ATL,2020,16,1604,12,13.868,0.75,0.87,-1.87,0.229,0.029,-0.28,-0.19,22,1.38,19.4202,1.21,2.58,172,10.75,0.113,0.414,0.429,0.186,0.09,-0.15,33,2.06,33.5177,2.09,0.52,255,15.94,0.131,0.171,0.200,0.086,-0.09,0.12,-11,-14.0975,3.0975,6737,421.1,0.481,0.343,0.7766,0.7824,-39.3618,-0.5843,7.5965,7279,0.7839,0.7756,60.3546,0.8292,7.2324,-99.7164,0.3641,26.16,38.98,-12.82,0.257,0.086,0.086,4.45,1.566,2.333,-0.767,3.90,9.28,-0.94,3.74,9.07,0.99,0.11,0.233,0.555,-0.056,0.224,0.543,0.059,0.007,5.83,12.93,2.87,6.48,8.75,2.09,0.03,0.349,0.774,0.172,0.388,0.524,0.125,0.002
3,ATL,2021,32,3171,31,37.171,0.97,1.16,-6.17,0.194,0.242,0.22,0.29,43,1.34,47.6095,1.49,-4.61,420,13.12,0.113,0.548,0.435,0.645,-0.04,0.28,57,1.78,58.9908,1.84,1.99,441,13.78,0.134,0.129,0.194,0.129,-0.28,-0.25,-14,-11.3813,-2.6187,16953,529.8,0.564,0.935,0.8098,0.8099,-1.7108,-0.0101,5.8928,13090,0.7477,0.7511,-43

## Focus Team

Isolate all seasons for `FOCUS_TEAM`. Change `FOCUS_TEAM` in Cell 1 to
repoint this section at any team in the dataset.


In [10]:
lou = team_stats[team_stats["team_name"] == FOCUS_TEAM].copy()
display(lou)

,team_name,season_name,count_games,minutes,points,xpoints,pts_per_game,xpts_per_game,points_overperformance,pts_per_game_pct,xpts_per_game_pct,pts_per_game_yoy,xpts_per_game_yoy,goals_for,goals_per_game,xgoals_for,xg_per_game,finishing_overperformance,shots_for,shots_per_game,xg_per_shot_for,xg_per_shot_for_pct,goals_per_game_pct,xg_per_game_pct,goals_per_game_yoy,xg_per_game_yoy,goals_against,goals_against_per_game,xgoals_against,xga_per_game,defensive_overperformance,shots_against,shots_against_per_game,xg_per_shot_against,xg_per_shot_against_pct,goals_against_per_game_pct,xga_per_game_pct,goals_against_per_game_yoy,xga_per_game_yoy,goal_difference,xgoal_difference,goal_difference_minus_xgoal_difference,attempted_passes_for,passes_per_game,pass_share,pass_share_pct,pass_completion_percentage_for,xpass_completion_percentage_for,passes_completed_over_expected_for,passes_completed_over_expected_p100_for,avg_vertical_distance_for,attempted_passes_against,pass_completion_percentage_against,xpass_completion_percentage_against,passes_completed_over_expected_against,passes_completed_over_expected_p100_against,avg_vertical_distance_against,passes_completed_over_expected_difference,avg_vertical_distance_difference,ga_for_total,ga_against_total,ga_difference,ga_for_total_pct,ga_against_total_pct,ga_difference_pct,ga_difference_yoy,ga_for_total_p96,ga_against_total_p96,ga_difference_p96,ga_for_passing,ga_for_receiving,ga_for_dribbling,ga_for_shooting,ga_for_interrupting,ga_for_fouling,ga_for_claiming,ga_for_passing_p96,ga_for_receiving_p96,ga_for_dribbling_p96,ga_for_shooting_p96,ga_for_interrupting_p96,ga_for_fouling_p96,ga_for_claiming_p96,ga_against_passing,ga_against_receiving,ga_against_dribbling,ga_against_shooting,ga_against_interrupting,ga_against_fouling,ga_against_claiming,ga_against_passing_p96,ga_against_receiving_p96,ga_against_dribbling_p96,ga_against_shooting_p96,ga_against_interrupting_p96,ga_against_fouling_p96,ga_against_claiming_p96
87,LOU,2017,32,3154,62,60.975,1.94,1.91,1.02,0.950,1.000,NaN,NaN,56,1.75,49.9881,1.56,6.01,501,15.66,0.100,0.350,0.917,0.917,NaN,NaN,30,0.94,25.4211,0.79,-4.58,291,9.09,0.087,0.933,0.900,0.967,NaN,NaN,26,24.5670,1.4330,16227,507.1,0.534,0.833,0.7546,0.7539,11.6333,0.0717,8.4556,14163,0.7200,0.7369,-239.4314,-1.6905,10.0252,251.0647,-1.5696,63.22,48.92,14.30,0.933,0.800,1.000,NaN,1.924,1.489,0.435,8.59,23.96,-0.81,11.82,16.23,3.54,-0.13,0.261,0.729,-0.025,0.360,0.494,0.108,-0.004,3.78,15.44,-3.37,6.26,24.12,2.46,0.22,0.115,0.470,-0.103,0.191,0.734,0.075,0.007
88,LOU,2018,34,3340,66,58.290,1.94,1.71,7.71,0.955,0.909,0.00,-0.20,68,2.00,55.2178,1.62,12.78,561,16.50,0.098,0.212,0.909,0.939,0.25,0.06,37,1.09,37.2168,1.09,0.22,373,10.97,0.100,0.591,0.788,0.909,0.15,0.30,31,18.0010,12.9990,18279,537.6,0.553,0.970,0.7684,0.7677,13.5596,0.0742,8.0708,14762,0.7070,0.7238,-248.8392,-1.6857,10.1486,262.3988,-2.0778,67.32,59.40,7.92,0.939,0.515,0.788,-6.38,1.935,1.707,0.228,10.56,26.15,-0.95,13.24,14.31,3.77,0.24,0.304,0.752,-0.027,0.381,0.411,0.108,0.007,8.41,19.56,-3.76,8.26,23.06,3.40,0.46,0.242,0.562,-0.108,0.237,0.663,0.098,0.013
89,LOU,2019,34,3352,60,62.861,1.76,1.85,-2.86,0.875,0.972,-0.18,0.14,55,1.62,65.0102,1.91,-10.01,554,16.29,0.117,0.736,0.625,0.944,-0.38,0.29,40,1.18,41.7580,1.23,1.76,390,11.47,0.107,0.681,0.792,0.847,0.09,0.14,15,23.2522,-8.2522,18225,536.0,0.578,1.000,0.7770,0.7658,204.4324,1.1217,7.4071,13294,0.6949,0.7127,-236.9692,-1.7825,10.7000,441.4016,-3.2928,73.67,62.84,10.83,0.917,0.472,0.833,2.91,2.110,1.800,0.310,9.34,29.85,3.45,12.95,15.56,2.66,-0.13,0.267,0.855,0.099,0.371,0.446,0.076,-0.004,6.36,20.76,0.54,8.90,23.61,2.47,0.20,0.182,0.595,0.015,0.255,0.676,0.071,0.006
90,LOU,2020,16,1610,35,27.184,2.19,1.70,7.82,0.957,0.771,0.43,-0.15,28,1.75,23.7207,1.48,4.28,241,15.06,0.098,0.071,0.757,0.600,0.13,-0.43,12,0.75,16.1993,1.01,4.20,153,9.56,0.106,0.814,0.929,0.943,-0.43,-0.22,16,7.5215,8.4785,8178,511.1,0.535,0.914,0.7860,0.7762,80.0695,0.9791,6.6772,7102,0.7525,0.7597,-51.3

## Usage Example

In [11]:
cols = [
    "team_name",
    "season_name",
    "count_games",
    "pts_per_game",
    "goals_per_game",
    "goals_against_per_game",
]
example = team_stats[cols].sort_values("pts_per_game", ascending=False)
example = example[example["count_games"] >= example["count_games"].max() / 2]
display(example.head(10))

,team_name,season_name,count_games,pts_per_game,goals_per_game,goals_against_per_game
95,LOU,2025,30,2.43,1.80,0.57
178,PHX,2019,34,2.29,2.53,1.00
28,CIN,2018,34,2.26,2.06,0.97
222,SA,2022,34,2.26,1.50,0.74
94,LOU,2024,34,2.24,2.44,1.21
264,TBR,2021,32,2.22,1.66,0.69
92,LOU,2022,34,2.12,1.88,0.82
180,PHX,2021,32,2.09,2.06,1.00
246,SLC,2017,32,2.09,1.84,0.91
25,CHS,2025,30,2.07,2.00,1.03
